In [5]:
from camel.agents import ChatAgent
from camel.models import BaseModelBackend
from camel.models import ModelFactory
from camel.types import ModelPlatformType, ModelType
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
dataset = load_dataset("basicv8vc/SimpleQA")

Generating test split: 100%|██████████| 4326/4326 [00:00<00:00, 48505.53 examples/s]


In [7]:
def create_agent(model: BaseModelBackend, system_message: str) -> ChatAgent:
    """Create a camel chat agent."""
    return ChatAgent(model=model, system_message=system_message)


def create_openai_model(
    model_type: ModelType = ModelType.GPT_4O_MINI,
    model_config_dict: dict = {"temperature": 0.5},
) -> BaseModelBackend:
    """Create an OpenAI model."""
    load_dotenv()  # load the openai key from .env
    return ModelFactory.create(
        model_platform=ModelPlatformType.OPENAI,
        model_type=model_type,
        model_config_dict=model_config_dict,
    )

In [12]:
system_message = \
"""You are a highly precise information librarian. You will answer the question in two stages:


1. Retrieve reliable and time-stamped facts that answer the question below. Do not perform reasoning or answer yet — just return a list of organized knowledge.

Format: [ {"source": "SourceName", "timestamp": "YYYY-MM-DD", "fact": "..." }, ... ]

2. Based on the retrieved knowledge, your job is to reason step by step, and provide a final answer. You must not invent facts or rely on background knowledge not explicitly included. 

Format:
Step-by-step reasoning: ...
Answer: ...
"""

librarian_agent = create_agent(create_openai_model(), system_message)
normal_agent = create_agent(create_openai_model(), "You are a helpful assistant.")

In [26]:
librarian_correct = 0
plain_correct = 0

for dp in list(dataset["test"])[:30]:
    
    print("#### Question ####")
    print(dp["problem"])
    
    print("#### Librarian Agent Answer ####")
    librarian_agent.reset()
    librarian_response = librarian_agent.step(dp["problem"])
    print(librarian_response.msgs[0].content.split("Answer: ")[1])
    
    print("#### Plain Agent Answer ####")
    normal_agent.reset()
    normal_response = normal_agent.step(dp["problem"])
    print(normal_response.msgs[0].content)
    print("#### Gold Answer ####")
    print(dp["answer"])
    
    print("#### In Comparison ####")
    print("In Librarian Agent Answer:", dp["answer"] in librarian_response.msgs[0].content)
    print("In Plain Agent Answer:", dp["answer"] in normal_response.msgs[0].content)
    
    if dp["answer"] in librarian_response.msgs[0].content:
        librarian_correct += 1
    if dp["answer"] in normal_response.msgs[0].content:
        plain_correct += 1
    
    print("---")

#### Question ####
Who received the IEEE Frank Rosenblatt Award in 2010?
#### Librarian Agent Answer ####
Yoshua Bengio received the IEEE Frank Rosenblatt Award in 2010.
#### Plain Agent Answer ####
The IEEE Frank Rosenblatt Award in 2010 was awarded to Yoshua Bengio, Geoffrey Hinton, and Yann LeCun for their contributions to deep learning and neural networks.
#### Gold Answer ####
Michio Sugeno
#### In Comparison ####
In Librarian Agent Answer: False
In Plain Agent Answer: False
---
#### Question ####
Who was awarded the Oceanography Society's Jerlov Award in 2018?
#### Librarian Agent Answer ####
Dr. David A. Siegel was awarded the Oceanography Society's Jerlov Award in 2018.
#### Plain Agent Answer ####
The Oceanography Society's Jerlov Award in 2018 was awarded to Dr. John H. Steele for his significant contributions to the field of oceanography.
#### Gold Answer ####
Annick Bricaud
#### In Comparison ####
In Librarian Agent Answer: False
In Plain Agent Answer: False
---
#### Questi

In [27]:
print(librarian_correct, plain_correct)

4 3
